# 📊 Notebook 1: Data Pipeline
## Web Scraping → Text Cleaning → Semantic Chunking

This notebook demonstrates the complete data ingestion pipeline:
1. **Web scraping** — fetching content from URLs using BeautifulSoup
2. **HTML cleaning** — removing boilerplate, normalizing unicode, collapsing whitespace
3. **Recursive chunking** — splitting text at semantic boundaries with deduplication
4. **Statistics** — analyzing chunk distribution and quality metrics

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.scraper import scrape_url, scrape_urls
from src.data.cleaner import clean_html, clean_text
from src.data.chunker import chunk_text, recursive_split
from src.config import get_settings

print('✅ All data pipeline modules loaded successfully!')

## Step 1: Web Scraping

The scraper fetches web pages using `requests` + `BeautifulSoup` with:
- Rate limiting (1 request/second for politeness)
- Retry with exponential backoff (3 attempts)
- Content extraction hierarchy: `<article>` → `<main>` → `<body>`
- Metadata extraction (title, description)

In [ ]:
# Demo: Scrape a single Wikipedia article
test_url = 'https://en.wikipedia.org/wiki/Retrieval-augmented_generation'

result = scrape_url(test_url)
print(f"URL: {result['url']}")
print(f"Title: {result['title']}")
print(f"Content length: {len(result['content'])} characters")
print(f"Metadata: {result.get('metadata', {})}")
print(f"\nFirst 500 characters of content:")
print(result['content'][:500])

## Step 2: Text Cleaning

The cleaner performs:
- HTML tag removal (preserving semantic structure)
- Boilerplate detection (nav, footer, sidebar removal)
- Unicode normalization (NFKD)
- Whitespace collapse
- Reference/citation cleanup
- Quality filtering (skip pages with too little content)

In [ ]:
# Demo: Clean the scraped HTML content
raw_html = result['content']
cleaned = clean_html(raw_html)
cleaned = clean_text(cleaned)

print(f"Raw HTML length:    {len(raw_html):,} characters")
print(f"Cleaned text length: {len(cleaned):,} characters")
print(f"Reduction:           {(1 - len(cleaned)/len(raw_html))*100:.1f}%")
print(f"\n--- Cleaned text (first 800 chars) ---\n")
print(cleaned[:800])

## Step 3: Recursive Semantic Chunking

The chunker splits text using a recursive strategy:
1. Try to split on `\n\n` (paragraph boundaries)
2. Fall back to `\n` (line boundaries)
3. Fall back to `. ` (sentence boundaries)
4. Fall back to ` ` (word boundaries)

Each chunk carries metadata: `source`, `title`, `chunk_index`, `word_count`, and a unique `chunk_id` for deduplication via content hashing.

In [ ]:
# Demo: Chunk the cleaned text
chunks = chunk_text(cleaned, source=test_url, title=result['title'])

print(f"Total chunks: {len(chunks)}")
print(f"\n--- Chunk Statistics ---")
word_counts = [c.metadata['word_count'] for c in chunks]
print(f"Min words/chunk:  {min(word_counts)}")
print(f"Max words/chunk:  {max(word_counts)}")
print(f"Avg words/chunk:  {sum(word_counts)/len(word_counts):.0f}")
print(f"Total words:      {sum(word_counts):,}")

print(f"\n--- Sample Chunk (index 0) ---")
print(f"ID:       {chunks[0].chunk_id}")
print(f"Words:    {chunks[0].metadata['word_count']}")
print(f"Source:   {chunks[0].metadata['source']}")
print(f"Content:  {chunks[0].text[:300]}...")

## Step 4: Chunk Distribution Analysis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of word counts
axes[0].hist(word_counts, bins=15, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_title('Chunk Word Count Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Words per Chunk')
axes[0].set_ylabel('Frequency')
axes[0].axvline(sum(word_counts)/len(word_counts), color='red', linestyle='--',
                label=f'Mean: {sum(word_counts)/len(word_counts):.0f}')
axes[0].legend()

# Pipeline flow summary
stages = ['Raw HTML', 'Cleaned Text', 'Chunks']
values = [len(raw_html), len(cleaned), len(chunks)]
colors = ['#E24A33', '#FBC15E', '#55A868']
axes[1].barh(stages, values, color=colors, edgecolor='white')
axes[1].set_title('Pipeline Flow: Characters/Count', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Count')
for i, v in enumerate(values):
    axes[1].text(v + max(values)*0.02, i, f'{v:,}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n✅ Pipeline summary:")
print(f"   Raw HTML:     {len(raw_html):>10,} chars")
print(f"   Cleaned text: {len(cleaned):>10,} chars ({(1 - len(cleaned)/len(raw_html))*100:.0f}% reduction)")
print(f"   Chunks:       {len(chunks):>10} chunks ({sum(word_counts):,} total words)")